# extract_audio_v2 — extração de áudio corrigida (A1–A5 da spec 10)

Substitui `extract_audio.ipynb` (v1, PRESERVADO — decisão D3). Correções:

- **A1 (máscara do padding):** a v1 fazia `pad_or_trim` até 30 s e promediava os
  1500 frames do encoder INCLUINDO o silêncio sintético — 34,5 % do vetor médio de
  treino era encoder de silêncio, e a fração variava com a duração (confound:
  o modelo podia ler a DURAÇÃO, não o áudio). Agora a média usa só os
  `n_valid = round(segundos_válidos × 50)` primeiros frames.
  ⚠ Ressalva a declarar na tese: a self-attention do encoder AINDA viu o padding
  dentro de cada janela; o que se remove é a diluição da MÉDIA, não o efeito do
  padding no encoder (que é o regime de treino do próprio Whisper).
- **A2 (janelas múltiplas):** a v1 CORTAVA em 30 s — 24,5 % do áudio de treino
  nunca chegava ao encoder (33,4 % dos vídeos passam de 30 s; o pior item de teste
  perdia 77,8 %). Agora o áudio é percorrido em janelas de 30 s e a média é
  ponderada pelos frames válidos de cada janela — matematicamente IGUAL à média
  sobre todos os frames válidos do áudio inteiro (não é heurística).
- **A3:** resolvido de graça por A1+A2 (o shift treino 0,655 vs teste 0,582 de
  `valid_fraction` deixa de existir como variável).
- **A4 = V2+V6+V7 do áudio:** falha → `_missing.json` (nunca `torch.zeros(1280)`
  no disco); `SPLIT` único + caminhos derivados + `_extraction_meta.json`;
  árvore `audio_features/v2/{split}/`.
- **A5:** ramo `else` no-op removido do loader mono.

O shape de saída continua [1280] float32 — o consumidor não muda de contrato.
Fontes dos números: analysis/audio_durations.csv, improvements/07, spec §A1–A3.

In [1]:
# P1-fix: os caminhos deste notebook sao RELATIVOS a EXIST_VIDEOS/ (plano 11 §3:
# "kernel do Jupyter em /home/a67606/Thesis/EXIST_VIDEOS ou cd ate la"). Se o kernel
# nascer noutro cwd, o assert do JSON da celula seguinte falha. Idempotente.
import os
from pathlib import Path

_TARGET = Path("/home/a67606/Thesis/EXIST_VIDEOS")
if Path.cwd() != _TARGET:
    assert _TARGET.is_dir(), f"diretorio do projeto nao encontrado: {_TARGET}"
    os.chdir(_TARGET)
assert Path("Dataset").is_dir(), f"cwd errado: {Path.cwd()} (sem Dataset/)"
print("cwd:", Path.cwd())


cwd: /home/a67606/Thesis/EXIST_VIDEOS


In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Extração de áudio v2 (Whisper encoder-only, média mascarada e janelada).
Ver célula 0 para o mapa de correções A1–A5. Salva [1280] float32 em
audio_features/v2/{split}/{id_EXIST}.pt.
"""
import json, platform, subprocess, sys, traceback
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchaudio
import torchaudio.transforms as T
import whisper
from tqdm import tqdm

# ── Config ───────────────────────────────────────────────────────────────────
SPLIT = "test"        # ← A ÚNICA LINHA A EDITAR: "training" ou "test" (V6/A4)

DATASET_JSON = {          # V6: nomes reais dos JSON não seguem o mesmo padrão
    "training": Path("./Dataset/training/EXIST2026_training.json"),
    "test":     Path("./Dataset/test/EXIST2026_test_clean.json"),
}
EXPECTED_N = {"training": 2524, "test": 674}

VIDEO_DIR     = Path(f"./Dataset/{SPLIT}/videos/")
OUTPUT_DIR    = Path(f"./audio_features/v2/{SPLIT}/")      # V7/A4: árvore versionada
JSON_PATH     = DATASET_JSON[SPLIT]
WHISPER_MODEL = "large-v3"                                 # NÃO MUDAR (spec §4)
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_SR     = 16000                                      # NÃO MUDAR (spec §4)

# Constantes do encoder do Whisper: janela de 30 s → 1500 frames → 50 frames/s.
CHUNK_SECONDS   = 30
CHUNK_SAMPLES   = CHUNK_SECONDS * TARGET_SR                # 480 000
N_ENC_FRAMES    = 1500
FRAMES_PER_SEC  = N_ENC_FRAMES // CHUNK_SECONDS            # 50
# ─────────────────────────────────────────────────────────────────────────────

assert SPLIT in DATASET_JSON, f"SPLIT inválido: {SPLIT!r}"
assert JSON_PATH.exists(), f"JSON não encontrado: {JSON_PATH}"
assert VIDEO_DIR.is_dir(), f"Diretório de vídeos não encontrado: {VIDEO_DIR}"
assert DEVICE == "cuda", "Sem GPU: a extração de áudio precisa de CUDA (pré-condição P0)"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Carregando Whisper {WHISPER_MODEL} em {DEVICE}...")
model = whisper.load_model(WHISPER_MODEL, device=DEVICE)
model.eval()

df = pd.read_json(JSON_PATH).transpose()
assert len(df) == EXPECTED_N[SPLIT], \
    f"JSON de {SPLIT} tem {len(df)} itens; esperado {EXPECTED_N[SPLIT]} — PARAR"
print(f"{SPLIT}: {len(df)} itens → {OUTPUT_DIR}")

resampler_cache = {}

Carregando Whisper large-v3 em cuda...
training: 2524 itens → audio_features/v2/training


In [3]:
def load_audio_mono_16k(video_path: Path):
    """Waveform [T] float32 mono a 16 kHz, ou None em falha (→ _missing.json, D4).
    A5: o ramo `else: waveform = waveform` (no-op) da v1 foi removido."""
    try:
        waveform, sr = torchaudio.load(str(video_path))        # [C, T]
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)      # [1, T]
        if sr != TARGET_SR:
            if sr not in resampler_cache:
                resampler_cache[sr] = T.Resample(orig_freq=sr, new_freq=TARGET_SR)
            waveform = resampler_cache[sr](waveform)
        return waveform.squeeze(0)                             # [T]
    except Exception as e:
        print(f"  [ERR] torchaudio.load falhou: {e}")
        return None


@torch.no_grad()
def extract_whisper_features_v2(waveform: torch.Tensor):
    """Encoder do Whisper com média MASCARADA (A1) e JANELADA (A2).

    A1 — dentro de cada janela de 30 s, a média usa só os primeiros
         n_valid = round(segundos_válidos × 50) frames do encoder; o resto é
         encoder de padding (silêncio sintético) e fica FORA da média.
    A2 — o áudio inteiro é percorrido em janelas de 30 s; a agregação final é a
         média das janelas PONDERADA por n_valid. Com pesos (n_valid_1, ...,
         n_valid_k) o resultado é exatamente a média sobre TODOS os frames
         válidos do áudio — o mesmo número que A1 daria se o Whisper não tivesse
         limite de 30 s (justificação matemática, spec §A2).

    Nota sobre o pseudocódigo da spec §A1: lá está `encoder_out[:n_valid]`, que
    fatiaria a dimensão de BATCH ([1,1500,1280]); o índice correto é
    `encoder_out[0, :n_valid]` (registrado no plano 11 §12).

    Devolve (features [1280] float32 em CPU, n_windows) ou (None, 0) se o
    waveform está vazio.
    """
    n_samples = int(waveform.shape[0])
    if n_samples == 0:
        return None, 0
    feats, weights = [], []
    for start in range(0, n_samples, CHUNK_SAMPLES):
        chunk = waveform[start:start + CHUNK_SAMPLES]
        valid_seconds = chunk.shape[0] / TARGET_SR
        n_valid = max(1, min(N_ENC_FRAMES, int(round(valid_seconds * FRAMES_PER_SEC))))
        audio = whisper.pad_or_trim(chunk.numpy())             # [480000]
        mel = whisper.log_mel_spectrogram(
            audio, n_mels=model.dims.n_mels).to(DEVICE)        # [n_mels, 3000]
        enc = model.encoder(mel.unsqueeze(0))                  # [1, 1500, 1280]
        feats.append(enc[0, :n_valid].mean(dim=0))             # A1: só frames válidos
        weights.append(float(n_valid))
    w = torch.tensor(weights, device=feats[0].device)
    features = (torch.stack(feats, dim=0) * w[:, None]).sum(dim=0) / w.sum()  # A2
    return features.cpu().float(), len(feats)

In [4]:
# ── Prova de que "média ponderada das janelas == média plana dos frames válidos".
# Teste puro de aritmética, sem encoder: se isto falhar, a implementação da
# agregação está errada e NADA deve ser extraído.
g = torch.Generator().manual_seed(0)
fake_frames = torch.randn(1500 + 1000, 1280, generator=g)   # 50 s de "frames válidos"
win1, win2 = fake_frames[:1500], fake_frames[1500:]         # janelas de 30 s e 20 s
w = torch.tensor([1500.0, 1000.0])
weighted = (torch.stack([win1.mean(0), win2.mean(0)]) * w[:, None]).sum(0) / w.sum()
flat = fake_frames.mean(0)
assert torch.allclose(weighted, flat, atol=1e-5), \
    "agregação ponderada != média plana — bug na célula 2, PARAR"
print(f"ok: max|Δ| = {(weighted - flat).abs().max():.2e} (esperado ~1e-7, float32)")

# n_valid: casos de borda
for secs, expect in [(30.0, 1500), (0.001, 1), (18.19, 910), (0.649, 32)]:
    nv = max(1, min(N_ENC_FRAMES, int(round(secs * FRAMES_PER_SEC))))
    assert nv == expect, (secs, nv, expect)
print("ok: n_valid nos casos de borda (30 s→1500, mediana treino 18,19 s→910, "
      "resto de 60,649 s→32, mínimo→1)")

ok: max|Δ| = 1.12e-08 (esperado ~1e-7, float32)
ok: n_valid nos casos de borda (30 s→1500, mediana treino 18,19 s→910, resto de 60,649 s→32, mínimo→1)


In [5]:
missing = []
window_hist = {}          # nº de janelas → contagem de itens (alimenta o teste 7)

for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Áudio v2 [{SPLIT}]"):
    item_id  = str(row["id_EXIST"])
    out_path = OUTPUT_DIR / f"{item_id}.pt"

    if out_path.exists():        # resume-safe — correto: árvore nasce vazia (V7)
        continue

    video_path = VIDEO_DIR / Path(str(row["path_video"])).name
    if not video_path.exists():
        # A4=V2/D4: NUNCA torch.save(torch.zeros(1280)) — ausência vira manifesto.
        missing.append({"id": item_id, "reason": f"vídeo não encontrado: {video_path.name}"})
        continue

    waveform = load_audio_mono_16k(video_path)
    if waveform is None:
        missing.append({"id": item_id, "reason": "torchaudio.load falhou"})
        continue

    try:
        features, n_windows = extract_whisper_features_v2(waveform)
    except Exception as e:
        traceback.print_exc()
        missing.append({"id": item_id, "reason": f"{type(e).__name__}: {e}"})
        continue

    if features is None:
        missing.append({"id": item_id, "reason": "waveform vazio (0 amostras)"})
        continue

    assert features.shape == (1280,), f"shape inesperado: {tuple(features.shape)}"
    window_hist[n_windows] = window_hist.get(n_windows, 0) + 1
    torch.save(features, out_path)

n_saved = len(list(OUTPUT_DIR.glob("*.pt")))
print(f"\n{SPLIT}: {n_saved} gravados | {len(missing)} faltantes")
print(f"histograma de janelas: {dict(sorted(window_hist.items()))}")
# Referência calculada de analysis/audio_durations.csv (durações ffprobe; a
# fronteira exata pode deslocar ±poucos itens porque aqui conta o waveform real):
#   training: {1: 1682, 2: 821, 3: 21}   → 842 itens com >1 janela
#   test:     {1: 512, 2: 145, 3: 15, 4: 1, 5: 1} → 162 itens com >1 janela
multi = sum(v for k, v in window_hist.items() if k > 1)
print(f"itens com >1 janela: {multi} "
      f"(esperado ~842 em training / ~162 em test, ±5)")

Áudio v2 [training]: 100%|███████████████████████████████████████████████████████████| 2524/2524 [08:45<00:00,  4.80it/s]


training: 2524 gravados | 0 faltantes
histograma de janelas: {1: 1689, 2: 813, 3: 22}
itens com >1 janela: 835 (esperado ~842 em training / ~162 em test, ±5)


In [6]:
def _pip_version(pkg: str) -> str:
    try:
        out = subprocess.run([sys.executable, "-m", "pip", "show", pkg],
                             capture_output=True, text=True).stdout
        return next(l.split(":", 1)[1].strip() for l in out.splitlines()
                    if l.startswith("Version:"))
    except Exception:
        return "desconhecida"

meta = {
    "split": SPLIT,
    "feature_shape": [1280],
    "dtype": "float32",
    "whisper_model": WHISPER_MODEL,
    "target_sr": TARGET_SR,
    "n_mels": int(model.dims.n_mels),
    "pooling": "média mascarada aos frames válidos (A1) sobre janelas de 30 s, "
               "ponderada por n_valid (A2); n_valid = round(segundos×50), clamp [1,1500]",
    "declared_caveat": "a self-attention do encoder ainda vê o padding dentro de cada "
                       "janela; removida a diluição da MÉDIA, não o efeito do padding "
                       "no encoder (spec §A1)",
    "n_items": int(len(df)),
    "n_saved": int(n_saved),
    "n_missing": int(len(missing)),
    "window_histogram": {str(k): v for k, v in sorted(window_hist.items())},
    "versions": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torchaudio": torchaudio.__version__,
        "openai_whisper": _pip_version("openai-whisper"),
        "numpy": np.__version__,
    },
    "date": datetime.now(timezone.utc).isoformat(),
    "notebook": "extract_audio_v2.ipynb",
}
with open(OUTPUT_DIR / "_extraction_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
with open(OUTPUT_DIR / "_missing.json", "w") as f:
    json.dump({"split": SPLIT, "missing": missing, "date": meta["date"]}, f, indent=2)

print(f"meta/manifesto gravados em {OUTPUT_DIR}")
if missing:
    print("⚠ A v1 teve 0 falhas de áudio nestes dados (0 sem stream, 0 ausentes — "
          "analysis/audio_durations.csv). QUALQUER faltante é regressão: investigar.")
    print("IDs:", [m["id"] for m in missing][:20])

meta/manifesto gravados em audio_features/v2/training
